# 1. Set the inputs
Import the pipeline and set the project root, outcome configuration, and private dataset path.

In [ ]:
from pathlib import Path

from postop_aki.config import load_experiment_config
from postop_aki.data import load_modeling_data, split_cohorts
from postop_aki.pipeline import evaluate_experiment, train_experiment

# These paths select the outcome and private input data.
PROJECT_ROOT = Path("..").resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "any_aki.yaml"
DATA_PATH = PROJECT_ROOT / "outcome_full_ml.csv"
config = load_experiment_config(CONFIG_PATH)

# 2. Validate the sample schema
Load the synthetic workbook, check its columns, and split the two cohorts.

In [ ]:
sample = load_modeling_data(PROJECT_ROOT / "data" / "sample.xlsx")
sample_cohorts = split_cohorts(sample)
{"columns": sample.shape[1], "internal": len(sample_cohorts.internal), "external": len(sample_cohorts.external)}

# 3. Load the study data
Load the private dataset and report the internal and external cohort sizes.

In [ ]:
data = load_modeling_data(DATA_PATH)
cohorts = split_cohorts(data)
{"internal": len(cohorts.internal), "external": len(cohorts.external)}

# 4. Train the models
Fit the three variants with stored parameters; leave the five-fold search disabled.

In [ ]:
# Set tune=True only to repeat the five-fold search.
trained_models = train_experiment(
    DATA_PATH, config, PROJECT_ROOT / "outputs" / "models", tune=False
)
trained_models

# 5. Evaluate the models
Evaluate both cohorts with a separate Youden threshold for each cohort.

In [ ]:
# Leave model_dir unset to use the checksum-verified artifacts.
results = evaluate_experiment(DATA_PATH, config, n_bootstrap=2_000)

# 6. Show the results
Round the aggregate metrics table to three decimal places.

In [ ]:
results.round(3)